In [1]:
import pandas as pd
import numpy as np

data = {
    "Student_ID": [101, 102, 103, 104, 105, 105, 106, 107, 108, 109],
    "Age": [20, 21, np.nan, 22, 20, 20, 21, 200, 22, 20],
    "Gender": [
        "Male", "Female", "female", "Male", "Male",
        "Male", "Female", "Male", "FEMALE", "Female"
    ],
    "Study_Hours": [2, 5, 3, np.nan, 4, 4, 6, 2, 8, 3],
    "Marks": [45, 78, 55, 60, 72, 72, np.nan, 90, 150, 48],
    "Department": [
        "CSE", "ECE", "CSE", "cse", "ECE",
        "ECE", "CSE", "EEE", "ece", "CSE"
    ],
    "Result": [
        "Fail", "Pass", "Fail", "Pass", "Pass",
        "Pass", "Pass", "Pass", "Pass", "Fail"
    ]
}

df = pd.DataFrame(data)

print(df)

   Student_ID    Age  Gender  Study_Hours  Marks Department Result
0         101   20.0    Male          2.0   45.0        CSE   Fail
1         102   21.0  Female          5.0   78.0        ECE   Pass
2         103    NaN  female          3.0   55.0        CSE   Fail
3         104   22.0    Male          NaN   60.0        cse   Pass
4         105   20.0    Male          4.0   72.0        ECE   Pass
5         105   20.0    Male          4.0   72.0        ECE   Pass
6         106   21.0  Female          6.0    NaN        CSE   Pass
7         107  200.0    Male          2.0   90.0        EEE   Pass
8         108   22.0  FEMALE          8.0  150.0        ece   Pass
9         109   20.0  Female          3.0   48.0        CSE   Fail


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# ---------------------------------------------------------
# 1. SAMPLE DATASET SETUP (Simulating your problematic data)
# ---------------------------------------------------------
data = {
    "Student_ID": [101, 102, 103, 104, 105, 105, 106, 107],
    "Age": [20, 22, np.nan, 200, 21, 21, 19, 23],  # Missing & Outlier (200)
    "Gender": ["Female", "female", "FEMALE", "Male", "Male", "Male", "Female", "Male"],  # Inconsistent
    "Department": ["CSE", "cse", "ECE", "CSE", "EEE", "EEE", "cse", "ECE"],  # Inconsistent
    "Study_Hours": [4.5, np.nan, 6.0, 2.0, 5.0, 5.0, 3.5, 8.0],  # Missing
    "Marks": [85, 90, 78, 150, np.nan, np.nan, 65, 95],  # Outlier (150) & Missing
    "Result": ["Pass", "Pass", "Pass", "Fail", "Pass", "Pass", "Fail", "Pass"],
}

df = pd.DataFrame(data)
print("=== ORIGINAL DATASET ===")
print(df)
print("\n" + "=" * 50 + "\n")

# ---------------------------------------------------------
# 2. STEP 1: REMOVE DUPLICATES
# ---------------------------------------------------------
df = df.drop_duplicates(subset=["Student_ID"]).reset_index(drop=True)

# ---------------------------------------------------------
# 3. STEP 2: STANDARDIZE INCONSISTENT CATEGORICAL VALUES
# ---------------------------------------------------------
string_cols = ["Gender", "Department", "Result"]
for col in string_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()

# ---------------------------------------------------------
# 4. STEP 3: HANDLE OUTLIERS (Flag invalid values as NaN)
# ---------------------------------------------------------
# Valid rules: Age between 15 and 60, Marks between 0 and 100
df.loc[(df["Age"] < 15) | (df["Age"] > 60), "Age"] = np.nan
df.loc[(df["Marks"] < 0) | (df["Marks"] > 100), "Marks"] = np.nan

# ---------------------------------------------------------
# 5. STEP 4: IMPUTE MISSING VALUES
# ---------------------------------------------------------
# Fill missing numerical values with their column Median
num_cols = ["Age", "Study_Hours", "Marks"]
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# ---------------------------------------------------------
# 6. STEP 5: ENCODE CATEGORICAL VARIABLES
# ---------------------------------------------------------
# Binary encoding for Result & Gender
df["Result"] = df["Result"].map({"PASS": 1, "FAIL": 0})
df["Gender"] = df["Gender"].map({"FEMALE": 1, "MALE": 0})

# One-Hot Encoding for Nominal variable (Department)
df = pd.get_dummies(df, columns=["Department"], prefix="Dept", dtype=int)

# ---------------------------------------------------------
# 7. STEP 6: FEATURE SCALING (Min-Max Normalization)
# ---------------------------------------------------------
scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ---------------------------------------------------------
# CLEANED DATASET RESULT
# ---------------------------------------------------------
print("=== CLEANED & PREPROCESSED DATASET ===")
print(df)

=== ORIGINAL DATASET ===
   Student_ID    Age  Gender Department  Study_Hours  Marks Result
0         101   20.0  Female        CSE          4.5   85.0   Pass
1         102   22.0  female        cse          NaN   90.0   Pass
2         103    NaN  FEMALE        ECE          6.0   78.0   Pass
3         104  200.0    Male        CSE          2.0  150.0   Fail
4         105   21.0    Male        EEE          5.0    NaN   Pass
5         105   21.0    Male        EEE          5.0    NaN   Pass
6         106   19.0  Female        cse          3.5   65.0   Fail
7         107   23.0    Male        ECE          8.0   95.0   Pass


=== CLEANED & PREPROCESSED DATASET ===
   Student_ID   Age  Gender  Study_Hours     Marks  Result  Dept_CSE  \
0         101  0.25       1     0.416667  0.666667       1         1   
1         102  0.75       1     0.458333  0.833333       1         1   
2         103  0.50       1     0.666667  0.433333       1         0   
3         104  0.50       0     0.000000  0

In [3]:
# Display first five rows
print(df.head())

# Number of rows and columns
print(df.shape)

# Column names
print(df.columns)

# Data types and non-null counts
df.info()

# Statistical summary of numerical columns
print(df.describe())

# Statistical summary including categorical columns
print(df.describe(include="all"))

# Number of unique values in each column
print(df.nunique())

# Check data types
print(df.dtypes)

   Student_ID   Age  Gender  Study_Hours     Marks  Result  Dept_CSE  \
0         101  0.25       1     0.416667  0.666667       1         1   
1         102  0.75       1     0.458333  0.833333       1         1   
2         103  0.50       1     0.666667  0.433333       1         0   
3         104  0.50       0     0.000000  0.666667       0         1   
4         105  0.50       0     0.500000  0.666667       1         0   

   Dept_ECE  Dept_EEE  
0         0         0  
1         0         0  
2         1         0  
3         0         0  
4         0         1  
(7, 9)
Index(['Student_ID', 'Age', 'Gender', 'Study_Hours', 'Marks', 'Result',
       'Dept_CSE', 'Dept_ECE', 'Dept_EEE'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Student_ID   7 non-null      int64  
 1   Age          7 non-null      float64
 2   Gende

In [4]:
import pandas as pd
import numpy as np

# Detect missing values per column
print(df.isnull().sum())

# Total percentage of missing values per column
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)

Student_ID     0
Age            0
Gender         0
Study_Hours    0
Marks          0
Result         0
Dept_CSE       0
Dept_ECE       0
Dept_EEE       0
dtype: int64
Student_ID     0.0
Age            0.0
Gender         0.0
Study_Hours    0.0
Marks          0.0
Result         0.0
Dept_CSE       0.0
Dept_ECE       0.0
Dept_EEE       0.0
dtype: float64


In [5]:
from numpy._core.defchararray import count
# Count missing values column-wise
print(df.isnull().sum())

# Calculate missing percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_percentage)

Student_ID     0
Age            0
Gender         0
Study_Hours    0
Marks          0
Result         0
Dept_CSE       0
Dept_ECE       0
Dept_EEE       0
dtype: int64
Student_ID     0.0
Age            0.0
Gender         0.0
Study_Hours    0.0
Marks          0.0
Result         0.0
Dept_CSE       0.0
Dept_ECE       0.0
Dept_EEE       0.0
dtype: float64


In [6]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# SAMPLE DATASET WITH MISSING VALUES
# ---------------------------------------------------------
data = {
    "Date": pd.date_range(start="2026-01-01", periods=6, freq="D"),
    "Temperature": [22.0, 24.0, np.nan, 28.0, np.nan, 30.0],  # Time-series
    "Salary": [50000, 52000, np.nan, 51000, 250000, 49000],  # Skewed / Outlier
    "Age": [25, 30, 28, np.nan, 32, 27],                      # Normal numerical
    "Department": ["IT", "HR", np.nan, "IT", "IT", "Finance"] # Categorical
}

df = pd.DataFrame(data)
print("=== ORIGINAL DATASET ===")
print(df)
print("\n" + "=" * 50 + "\n")

=== ORIGINAL DATASET ===
        Date  Temperature    Salary   Age Department
0 2026-01-01         22.0   50000.0  25.0         IT
1 2026-01-02         24.0   52000.0  30.0         HR
2 2026-01-03          NaN       NaN  28.0        NaN
3 2026-01-04         28.0   51000.0   NaN         IT
4 2026-01-05          NaN  250000.0  32.0         IT
5 2026-01-06         30.0   49000.0  27.0    Finance




In [10]:
import numpy as np
import pandas as pd

# 1. Create/Ensure df exists
data = {
    "Age": [20, 22, np.nan, 21, 19],
    "Study_Hours": [4.5, np.nan, 6.0, 5.0, 3.5],
    "Marks": [85, 90, 78, np.nan, 65],
}
df = pd.DataFrame(data)

# 2. Work on a copy of the raw data
df_clean = df.copy()

# 3. Fill missing numerical values with column medians
numeric_cols = ["Age", "Study_Hours", "Marks"]

for col in numeric_cols:
    # Safely convert to numeric (turns invalid strings into NaN)
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # Compute median and fill missing values
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

# 4. Check missing counts
print("Missing values per column:")
print(df_clean[numeric_cols].isnull().sum())

print("\nCleaned DataFrame:")
print(df_clean)

Missing values per column:
Age            0
Study_Hours    0
Marks          0
dtype: int64

Cleaned DataFrame:
    Age  Study_Hours  Marks
0  20.0         4.50   85.0
1  22.0         4.75   90.0
2  20.5         6.00   78.0
3  21.0         5.00   81.5
4  19.0         3.50   65.0


In [12]:
import pandas as pd

# Sample dataset with duplicate records
data = {
    "Student_ID": [101, 102, 103, 104, 105, 105, 106],
    "Name": ["Rahul", "Priya", "Amit", "Sneha", "Vikas", "Vikas", "Ananya"],
    "Age": [20, 22, 21, 20, 21, 21, 19],
    "Marks": [85, 90, 78, 88, 75, 75, 92]
}

df = pd.DataFrame(data)

In [13]:
print("Duplicate rows:", df_clean.duplicated().sum())

print(df_clean[df_clean.duplicated(keep=False)])

Duplicate rows: 0
Empty DataFrame
Columns: [Age, Study_Hours, Marks]
Index: []


In [14]:
df_clean = df_clean.drop_duplicates()

print("Shape after removing duplicates:", df_clean.shape)

Shape after removing duplicates: (5, 3)


In [17]:
import pandas as pd

# Sample dataset with inconsistent text entries and extra whitespace
data = {
    "Student_ID": [101, 102, 103, 104, 105, 106, 107],
    "Department": ["CSE", " cse", "Cse ", "ECE", "ece", "MECH", " ECE "],
    "Gender": ["Female", "female", "FEMALE", "Male", " male", "MALE", "Female "]
}

df = pd.DataFrame(data)

In [19]:
import pandas as pd

# 1. Clean column headers by stripping leading/trailing whitespace
df_clean.columns = df_clean.columns.str.strip()

# 2. List target categorical columns
categorical_cols = ["Gender", "Department", "Result"]

# 3. Print value counts safely for each column that exists
for col in categorical_cols:
    if col in df_clean.columns:
        print(f"=== {col} Value Counts (including NaNs) ===")
        print(df_clean[col].value_counts(dropna=False))
        print("-" * 40)
    else:
        print(f"Warning: Column '{col}' not found in DataFrame!")

In [26]:
import numpy as np
import pandas as pd

# 1. Create sample dataset
data = {
    "Gender": ["Female ", "female", "FEMALE", "Male", " male", np.nan],
    "Department": ["CSE", " cse", "Cse ", "ECE", "ece", "MECH"],
    "Result": ["Pass", "pass ", "PASS", "Fail", " fail", "Pass"],
}

df_clean = pd.DataFrame(data)

# 2. Clean text columns (strip whitespace + uppercase)
for col in ["Gender", "Department", "Result"]:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.upper()

# 3. Print unique values and value counts
print("=== UNIQUE VALUES ===")
print("Gender:", list(df_clean["Gender"].unique()))
print("Department:", list(df_clean["Department"].unique()))
print("Result:", list(df_clean["Result"].unique()))

print("\n=== VALUE COUNTS ===")
for col in ["Gender", "Department", "Result"]:
    print(f"\n--- {col} ---")
    print(df_clean[col].value_counts(dropna=False))

=== UNIQUE VALUES ===
Gender: ['FEMALE', 'MALE', 'NAN']
Department: ['CSE', 'ECE', 'MECH']
Result: ['PASS', 'FAIL']

=== VALUE COUNTS ===

--- Gender ---
Gender
FEMALE    3
MALE      2
NAN       1
Name: count, dtype: int64

--- Department ---
Department
CSE     3
ECE     2
MECH    1
Name: count, dtype: int64

--- Result ---
Result
PASS    4
FAIL    2
Name: count, dtype: int64


In [27]:
df_clean["Department"] = (
    df_clean["Department"].str.upper()
)

print(df_clean["Department"].unique())

['CSE' 'ECE' 'MECH']


In [28]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. SAMPLE DATASET SETUP
# ---------------------------------------------------------
data = {
    "Student_ID": [101, 102, 103, 104, 105, 106, 107, 108],
    "Age": [20, 21, 22, 200, 19, 23, 21, 20],            # 200 is an invalid error
    "Study_Hours": [4, 5, 3, 10, 4, 5, 2, 4],            # 10 is valid extreme value
    "Marks": [85, 90, 78, 150, 65, 88, 72, 95]           # 150 exceeds max limit of 100
}

df = pd.DataFrame(data)
df_clean = df.copy()

# ---------------------------------------------------------
# STEP 7.1: DETECT OUTLIERS USING THE IQR METHOD
# ---------------------------------------------------------
def detect_iqr_outliers(data_frame, column):
    Q1 = data_frame[column].quantile(0.25)
    Q3 = data_frame[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    outliers = data_frame[(data_frame[column] < lower_bound) | (data_frame[column] > upper_bound)]

    print(f"=== IQR Outlier Report for '{column}' ===")
    print(f"Q1 (25th percentile): {Q1}")
    print(f"Q3 (75th percentile): {Q3}")
    print(f"IQR: {IQR}")
    print(f"Valid Range (Lower to Upper Fence): [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"Number of Outliers Detected: {len(outliers)}")
    if len(outliers) > 0:
        print("Outlier Rows:")
        print(outliers[["Student_ID", column]])
    print("-" * 50)

    return lower_bound, upper_bound

# Run detection for numerical columns
for col in ["Age", "Marks", "Study_Hours"]:
    detect_iqr_outliers(df_clean, col)

=== IQR Outlier Report for 'Age' ===
Q1 (25th percentile): 20.0
Q3 (75th percentile): 22.25
IQR: 2.25
Valid Range (Lower to Upper Fence): [16.62, 25.62]
Number of Outliers Detected: 1
Outlier Rows:
   Student_ID  Age
3         104  200
--------------------------------------------------
=== IQR Outlier Report for 'Marks' ===
Q1 (25th percentile): 76.5
Q3 (75th percentile): 91.25
IQR: 14.75
Valid Range (Lower to Upper Fence): [54.38, 113.38]
Number of Outliers Detected: 1
Outlier Rows:
   Student_ID  Marks
3         104    150
--------------------------------------------------
=== IQR Outlier Report for 'Study_Hours' ===
Q1 (25th percentile): 3.75
Q3 (75th percentile): 5.0
IQR: 1.25
Valid Range (Lower to Upper Fence): [1.88, 6.88]
Number of Outliers Detected: 1
Outlier Rows:
   Student_ID  Study_Hours
3         104           10
--------------------------------------------------


In [29]:
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower) |
        (data[column] > upper)
    ]

    print(f"\nColumn: {column}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower fence:", lower)
    print("Upper fence:", upper)
    print("Outlier rows:\n", outliers)

for col in ["Age", "Study_Hours", "Marks"]:
    detect_outliers_iqr(df_clean, col)


Column: Age
Q1: 20.0
Q3: 22.25
IQR: 2.25
Lower fence: 16.625
Upper fence: 25.625
Outlier rows:
    Student_ID  Age  Study_Hours  Marks
3         104  200           10    150

Column: Study_Hours
Q1: 3.75
Q3: 5.0
IQR: 1.25
Lower fence: 1.875
Upper fence: 6.875
Outlier rows:
    Student_ID  Age  Study_Hours  Marks
3         104  200           10    150

Column: Marks
Q1: 76.5
Q3: 91.25
IQR: 14.75
Lower fence: 54.375
Upper fence: 113.375
Outlier rows:
    Student_ID  Age  Study_Hours  Marks
3         104  200           10    150


In [30]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# SAMPLE DATASET WITH INVALID DATA ENTRY ERRORS
# ---------------------------------------------------------
data = {
    "Student_ID": [101, 102, 103, 104, 105],
    "Age": [20, 22, 200, 21, 19],       # 200 is an invalid age
    "Marks": [85, 150, 78, 90, 65]       # 150 exceeds max mark limit of 100
}

df_clean = pd.DataFrame(data)

print("=== BEFORE CORRECTION ===")
print(df_clean)
print("\n" + "=" * 40 + "\n")

# ---------------------------------------------------------
# STEP 7.2: IDENTIFY AND CORRECT INVALID VALUES
# ---------------------------------------------------------

# 1. Convert domain rule violations to NaN (Missing Values)
# Valid constraints: Age between 15 and 60; Marks between 0 and 100
df_clean.loc[(df_clean["Age"] < 15) | (df_clean["Age"] > 60), "Age"] = np.nan
df_clean.loc[(df_clean["Marks"] < 0) | (df_clean["Marks"] > 100), "Marks"] = np.nan

print("=== AFTER CONVERTING INVALID VALUES TO NaN ===")
print(df_clean)
print("\n" + "=" * 40 + "\n")

# 2. Impute the converted NaNs with Column Medians
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Marks"] = df_clean["Marks"].fillna(df_clean["Marks"].median())

print("=== FINAL CLEANED DATASET ===")
print(df_clean)

=== BEFORE CORRECTION ===
   Student_ID  Age  Marks
0         101   20     85
1         102   22    150
2         103  200     78
3         104   21     90
4         105   19     65


=== AFTER CONVERTING INVALID VALUES TO NaN ===
   Student_ID   Age  Marks
0         101  20.0   85.0
1         102  22.0    NaN
2         103   NaN   78.0
3         104  21.0   90.0
4         105  19.0   65.0


=== FINAL CLEANED DATASET ===
   Student_ID   Age  Marks
0         101  20.0   85.0
1         102  22.0   81.5
2         103  20.5   78.0
3         104  21.0   90.0
4         105  19.0   65.0


In [32]:
import numpy as np
import pandas as pd

# Work on a copy of the dataframe
df_clean = df.copy()

# 1. Ensure numeric data types (converts any residual text to NaN safely)
numeric_cols = ["Age", "Study_Hours", "Marks"]
for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# 2. Flag invalid domain values as NaN
if "Age" in df_clean.columns:
    df_clean.loc[~df_clean["Age"].between(15, 60), "Age"] = np.nan

if "Marks" in df_clean.columns:
    df_clean.loc[~df_clean["Marks"].between(0, 100), "Marks"] = np.nan

# 3. Refill invalid/missing values using column medians
for col in numeric_cols:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

# 4. Display cleaned numerical columns safely
available_cols = [c for c in numeric_cols if c in df_clean.columns]
print(df_clean[available_cols])

    Age  Study_Hours  Marks
0  20.0            4   85.0
1  21.0            5   90.0
2  22.0            3   78.0
3  21.0           10   85.0
4  19.0            4   65.0
5  23.0            5   88.0
6  21.0            2   72.0
7  20.0            4   95.0


In [34]:
import pandas as pd
import numpy as np

# 1. Sample Cleaned Data
data = {
    "Student_ID": [101, 102, 103, 104, 105],
    "Gender": ["FEMALE", "FEMALE", "MALE", "MALE", "FEMALE"],
    "Department": ["CSE", "ECE", "CSE", "MECH", "EEE"],
    "Result": ["PASS", "PASS", "FAIL", "PASS", "FAIL"]
}

df_clean = pd.DataFrame(data)

# ---------------------------------------------------------
# SAFE ENCODING WORKFLOW
# ---------------------------------------------------------

# Step 1: Ensure text data is string, trimmed, and uppercase before mapping
for col in ["Gender", "Department", "Result"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.upper()

# Step 2: Binary Mapping for Gender and Result (Safe fallback with map + fillna)
gender_map = {"FEMALE": 1, "MALE": 0}
result_map = {"PASS": 1, "FAIL": 0}

if "Gender" in df_clean.columns:
    df_clean["Gender"] = df_clean["Gender"].map(gender_map).fillna(0).astype(int)

if "Result" in df_clean.columns:
    df_clean["Result"] = df_clean["Result"].map(result_map).fillna(0).astype(int)

# Step 3: One-Hot Encoding for Department (Pandas get_dummies)
if "Department" in df_clean.columns:
    df_clean = pd.get_dummies(df_clean, columns=["Department"], prefix="Dept", dtype=int)

print("=== FINAL ENCODED DATASET ===")
print(df_clean)

=== FINAL ENCODED DATASET ===
   Student_ID  Gender  Result  Dept_CSE  Dept_ECE  Dept_EEE  Dept_MECH
0         101       1       1         1         0         0          0
1         102       1       1         0         1         0          0
2         103       0       0         1         0         0          0
3         104       0       1         0         0         0          1
4         105       1       0         0         0         1          0


In [35]:
df_clean["Result"] = df_clean["Result"].map({
    "Fail": 0,
    "Pass": 1
})

print(df_clean[["Student_ID", "Result"]])

   Student_ID  Result
0         101     NaN
1         102     NaN
2         103     NaN
3         104     NaN
4         105     NaN


In [37]:
import numpy as np
import pandas as pd

# 1. Strip extra whitespace from column names to prevent KeyError
df_clean.columns = df_clean.columns.str.strip()

# 2. Ensure target columns exist and clean text entries
target_cols = ["Gender", "Department"]
for col in target_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().str.upper()

# 3. Safe One-Hot Encoding
df_encoded = pd.get_dummies(
    df_clean,
    columns=[col for col in target_cols if col in df_clean.columns],
    drop_first=True,  # Avoids multicollinearity
    dtype=int,
)

# 4. Display result
print("=== FIRST 5 ROWS ===")
print(df_encoded.head())

print("\n=== ENCODED COLUMNS ===")
print(df_encoded.columns.tolist())

=== FIRST 5 ROWS ===
   Student_ID  Result  Dept_CSE  Dept_ECE  Dept_EEE  Dept_MECH  Gender_1
0         101     NaN         1         0         0          0         1
1         102     NaN         0         1         0          0         1
2         103     NaN         1         0         0          0         0
3         104     NaN         0         0         0          1         0
4         105     NaN         0         0         1          0         1

=== ENCODED COLUMNS ===
['Student_ID', 'Result', 'Dept_CSE', 'Dept_ECE', 'Dept_EEE', 'Dept_MECH', 'Gender_1']


In [38]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 1. Sample Cleaned Dataset
data = {
    "Student_ID": [101, 102, 103, 104, 105],
    "Age": [20, 22, 21, 21, 19],
    "Study_Hours": [4.5, 5.0, 6.0, 5.0, 3.5],
    "Marks": [85, 90, 78, 82, 65]
}

df_clean = pd.DataFrame(data)
num_cols = ["Age", "Study_Hours", "Marks"]

# =========================================================
# METHOD 1: Standardization (StandardScaler)
# Formula: z = (x - mean) / std_dev
# Result: Mean = 0, Standard Deviation = 1
# =========================================================
scaler_standard = StandardScaler()

df_standard = df_clean.copy()
df_standard[num_cols] = scaler_standard.fit_transform(df_clean[num_cols])

print("=== METHOD 1: STANDARDSCALER (Mean=0, Std=1) ===")
print(df_standard)
print("\n" + "=" * 50 + "\n")

# =========================================================
# METHOD 2: Normalization (MinMaxScaler)
# Formula: x_scaled = (x - min) / (max - min)
# Result: Values scaled strictly between [0, 1]
# =========================================================
scaler_minmax = MinMaxScaler()

df_minmax = df_clean.copy()
df_minmax[num_cols] = scaler_minmax.fit_transform(df_clean[num_cols])

print("=== METHOD 2: MINMAXSCALER (Bounded between [0, 1]) ===")
print(df_minmax)

=== METHOD 1: STANDARDSCALER (Mean=0, Std=1) ===
   Student_ID       Age  Study_Hours     Marks
0         101 -0.588348    -0.369274  0.590899
1         102  1.372813     0.246183  1.181799
2         103  0.392232     1.477098 -0.236360
3         104  0.392232     0.246183  0.236360
4         105 -1.568929    -1.600189 -1.772698


=== METHOD 2: MINMAXSCALER (Bounded between [0, 1]) ===
   Student_ID       Age  Study_Hours  Marks
0         101  0.333333          0.4   0.80
1         102  1.000000          0.6   1.00
2         103  0.666667          1.0   0.52
3         104  0.666667          0.6   0.68
4         105  0.000000          0.0   0.00


In [43]:
print("RAW DATA")
print(df.head())

print("\nCLEANED DATA")
print(df_clean.head())

print("\nENCODED DATA")
print(df_encoded.head())

print("\nSCALED DATA")
print(df_scaled.head())

RAW DATA
   Student_ID  Age  Study_Hours  Marks
0         101   20            4     85
1         102   21            5     90
2         103   22            3     78
3         104  200           10    150
4         105   19            4     65

CLEANED DATA
   Student_ID  Age  Study_Hours  Marks
0         101   20          4.5     85
1         102   22          5.0     90
2         103   21          6.0     78
3         104   21          5.0     82
4         105   19          3.5     65

ENCODED DATA
   Student_ID  Result  Dept_CSE  Dept_ECE  Dept_EEE  Dept_MECH  Gender_1
0         101     NaN         1         0         0          0         1
1         102     NaN         0         1         0          0         1
2         103     NaN         1         0         0          0         0
3         104     NaN         0         0         0          1         0
4         105     NaN         0         0         1          0         1

SCALED DATA
   Student_ID  Result  Dept_CSE  Dept_ECE  D